# PyPDF Tutorial: Text Extraction from PDFs
Learn how to extract text from PDF files using PyPDF library. This tutorial covers loading PDFs, extracting text, handling multiple pages, and cleaning extracted text.

## 1) Install PyPDF Library
We'll use PyPDF to read and extract text from PDF files. This library is already installed, but here's how to install it if needed.

In [ ]:
%pip install pypdf

## 2) Import Required Libraries
Import PyPDF and utilities for file handling and text processing.

In [1]:
from pypdf import PdfReader, PdfWriter
import os
from pathlib import Path
import re

# For this tutorial, we'll use a sample PDF path
# In practice, you would replace this with your actual PDF file path
sample_pdf_path = "Earthquakes.pdf"  # Replace with your PDF file path

## 3) Load a PDF File
Load a PDF file from disk using PdfReader. The PdfReader object provides access to all pages and metadata of the PDF.

In [2]:
# Example: Load a PDF file (replace with your actual PDF path)
# This is a demonstration - you need to have an actual PDF file
try:
    pdf_reader = PdfReader(sample_pdf_path)
    print(f"PDF loaded successfully: {sample_pdf_path}")
    print(f"Total number of pages: {len(pdf_reader.pages)}")
except FileNotFoundError:
    print(f"Note: PDF file '{sample_pdf_path}' not found.")
    print("In practice, you would do this:")
    print("  pdf_reader = PdfReader('your_pdf_file.pdf')")
except Exception as e:
    print(f"Error loading PDF: {e}")

PDF loaded successfully: Earthquakes.pdf
Total number of pages: 34


## 4) Extract Text from All Pages
Iterate through all pages in the PDF and extract text from each page using the `extract_text()` method. This is the primary way to get text content from a PDF.

In [5]:
# Function to extract text from all pages
def extract_all_text(pdf_path):
    """Extract text from all pages of a PDF file."""
    try:
        pdf_reader = PdfReader(pdf_path)
        all_text = []
        
        for page_num, page in enumerate(pdf_reader.pages):
            # Extract text from the page
            text = page.extract_text()
            all_text.append({
                'page_number': page_num + 1,
                'text': text
            })
            print(f"Extracted text from page {page_num + 1}")
        
        return all_text
    except FileNotFoundError:
        print(f"PDF file '{pdf_path}' not found.")
        return []
    except Exception as e:
        print(f"Error extracting text: {e}")
        return []

# Example usage
# text_data = extract_all_text(sample_pdf_path)
# if text_data:
#     for page_info in text_data:
#         print(f"\n--- Page {page_info['page_number']} ---")
#         print(page_info['text'][:200])  # Print first 200 characters

## 5) Extract Text from Specific Pages
Extract text from individual pages or specific page ranges. This is useful when you only need text from certain pages.

In [6]:
# Function to extract text from specific pages
def extract_specific_pages(pdf_path, page_numbers):
    """
    Extract text from specific pages.
    
    Args:
        pdf_path: Path to the PDF file
        page_numbers: List of page numbers (1-indexed) or a range
    
    Returns:
        Dictionary with page numbers and extracted text
    """
    try:
        pdf_reader = PdfReader(pdf_path)
        extracted_pages = {}
        
        for page_num in page_numbers:
            # Pages are 0-indexed in PdfReader, but we use 1-indexed for user
            if 1 <= page_num <= len(pdf_reader.pages):
                page = pdf_reader.pages[page_num - 1]
                text = page.extract_text()
                extracted_pages[page_num] = text
                print(f"Extracted text from page {page_num}")
            else:
                print(f"Page {page_num} is out of range (Total pages: {len(pdf_reader.pages)})")
        
        return extracted_pages
    except FileNotFoundError:
        print(f"PDF file '{pdf_path}' not found.")
        return {}
    except Exception as e:
        print(f"Error extracting specific pages: {e}")
        return {}

# Example usage: Extract pages 1, 3, and 5
# page_data = extract_specific_pages(sample_pdf_path, [1, 3, 5])

# Or extract a range of pages (e.g., pages 1 to 5)
# page_range = list(range(1, 6))  # Pages 1 through 5
# page_data = extract_specific_pages(sample_pdf_path, page_range)

## 6) Handle Multi-page PDFs
Get the total number of pages and process large PDFs efficiently. This is important for batch processing and memory-efficient extraction.

In [7]:
# Function to efficiently process large PDFs
def process_large_pdf(pdf_path, batch_size=10):
    """
    Process a large PDF file in batches to manage memory efficiently.
    
    Args:
        pdf_path: Path to the PDF file
        batch_size: Number of pages to process at a time
    
    Returns:
        Generator yielding batches of extracted text
    """
    try:
        pdf_reader = PdfReader(pdf_path)
        total_pages = len(pdf_reader.pages)
        print(f"Total pages in PDF: {total_pages}")
        
        for start_page in range(0, total_pages, batch_size):
            batch = []
            end_page = min(start_page + batch_size, total_pages)
            
            for page_idx in range(start_page, end_page):
                page = pdf_reader.pages[page_idx]
                text = page.extract_text()
                batch.append({
                    'page_number': page_idx + 1,
                    'text': text
                })
            
            print(f"Processed pages {start_page + 1} to {end_page}")
            yield batch
            
    except FileNotFoundError:
        print(f"PDF file '{pdf_path}' not found.")
    except Exception as e:
        print(f"Error processing PDF: {e}")

# Example usage: Process PDF in batches of 5 pages
# for batch in process_large_pdf(sample_pdf_path, batch_size=5):
#     for page_info in batch:
#         print(f"Page {page_info['page_number']}: {len(page_info['text'])} characters")

## 7) Clean and Process Extracted Text
Apply text cleaning techniques like removing extra whitespace, line breaks, and special characters to normalize extracted text for better processing.

In [ ]:
# Function to clean extracted text
def clean_text(text):
    """
    Clean extracted text by removing extra whitespace and normalizing content.
    
    Args:
        text: Raw text extracted from PDF
    
    Returns:
        Cleaned text
    """
    if not text:
        return ""
    
    # Remove multiple spaces and replace with single space
    text = re.sub(r' +', ' ', text)
    
    # Remove multiple newlines and replace with single newline
    text = re.sub(r'\n\n+', '\n', text)
    
    # Remove leading/trailing whitespace from each line
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

# Complete workflow: Extract and clean text
def extract_and_clean_pdf(pdf_path):
    """
    Complete workflow: Extract text from PDF and clean it.
    
    Args:
        pdf_path: Path to the PDF file
    
    Returns:
        List of cleaned text from each page
    """
    try:
        pdf_reader = PdfReader(pdf_path)
        cleaned_texts = []
        
        for page_num, page in enumerate(pdf_reader.pages):
            raw_text = page.extract_text()
            cleaned_text = clean_text(raw_text)
            cleaned_texts.append({
                'page': page_num + 1,
                'raw_length': len(raw_text),
                'cleaned_length': len(cleaned_text),
                'text': cleaned_text
            })
            print(f"Page {page_num + 1}: {len(raw_text)} -> {len(cleaned_text)} characters")
        
        return cleaned_texts
    except FileNotFoundError:
        print(f"PDF file '{pdf_path}' not found.")
        return []
    except Exception as e:
        print(f"Error extracting and cleaning PDF: {e}")
        return []

# Example usage
# cleaned_data = extract_and_clean_pdf(sample_pdf_path)
# if cleaned_data:
#     print("\nCleaned text from first page:")
#     print(cleaned_data[0]['text'])

Page 1: 177 -> 175 characters
Page 2: 415 -> 410 characters
Page 3: 391 -> 388 characters
Page 4: 142 -> 140 characters
Page 5: 143 -> 141 characters
Page 6: 345 -> 339 characters
Page 7: 562 -> 559 characters
Page 8: 341 -> 336 characters
Page 9: 403 -> 399 characters
Page 10: 224 -> 223 characters
Page 11: 371 -> 364 characters
Page 12: 402 -> 395 characters
Page 13: 292 -> 286 characters
Page 14: 227 -> 222 characters
Page 15: 589 -> 583 characters
Page 16: 459 -> 452 characters
Page 17: 416 -> 410 characters
Page 18: 676 -> 671 characters
Page 19: 718 -> 713 characters
Page 20: 244 -> 241 characters
Page 21: 278 -> 272 characters
Page 22: 400 -> 391 characters
Page 23: 410 -> 404 characters
Page 24: 326 -> 319 characters
Page 25: 290 -> 283 characters
Page 26: 280 -> 279 characters
Page 27: 462 -> 455 characters
Page 28: 357 -> 350 characters
Page 29: 284 -> 281 characters
Page 30: 232 -> 228 characters
Page 31: 412 -> 404 characters
Page 32: 577 -> 568 characters
Page 33: 468 -> 4

## Bonus: Practical Example - Extracting and Saving to File
Combine all the techniques to extract text from a PDF and save it to a text file for later use (useful for your RAG system).

In [ ]:
# Complete practical example: Extract PDF text and save to file
def extract_pdf_to_file(pdf_path, output_file=None):
    """
    Extract all text from PDF and save to a text file.
    
    Args:
        pdf_path: Path to the PDF file
        output_file: Path to save the output (if None, creates one automatically)
    
    Returns:
        Path to the output file
    """
    try:
        if output_file is None:
            output_file = Path(pdf_path).stem + "_extracted.txt"
        
        pdf_reader = PdfReader(pdf_path)
        total_pages = len(pdf_reader.pages)
        
        with open(output_file, 'w', encoding='utf-8') as f:
            for page_num, page in enumerate(pdf_reader.pages):
                raw_text = page.extract_text()
                cleaned_text = clean_text(raw_text)
                
                # Write page header
                f.write(f"\n{'='*50}\n")
                f.write(f"PAGE {page_num + 1}\n")
                f.write(f"{'='*50}\n\n")
                
                # Write cleaned text
                f.write(cleaned_text)
                f.write("\n")
                
                print(f"Processed page {page_num + 1}/{total_pages}")
        
        print(f"\nExtraction complete! Text saved to: {output_file}")
        return output_file
        
    except FileNotFoundError:
        print(f"PDF file '{pdf_path}' not found.")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

# Example usage for your RAG system:
# extract_pdf_to_file("your_document.pdf", "extracted_text.txt")
# 
# Now you can use this extracted text to:
# 1. Generate embeddings with sentence-transformers
# 2. Store vectors in your PostgreSQL database
# 3. Perform semantic search with your RAG system

print("PyPDF tutorial complete!")
print("\nKey functions learned:")
print("- extract_all_text(): Extract text from all pages")
print("- extract_specific_pages(): Extract specific pages")
print("- process_large_pdf(): Efficiently handle large PDFs")
print("- clean_text(): Clean extracted text")
print("- extract_and_clean_pdf(): Combined workflow")
print("- extract_pdf_to_file(): Save extracted text to file")